# API de OpenAI — Notebook interactivo

Chat completions, streaming, function calling, embeddings y moderación con la API de OpenAI.

In [ ]:
# pip install openai python-dotenv
import os
from openai import OpenAI

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
MODELO_CHAT = 'gpt-4o-mini'
MODELO_EMBED = 'text-embedding-3-small'
print(f'Cliente OpenAI listo — modelo: {MODELO_CHAT}')

## 1. Chat completions básico

In [ ]:
def chat(mensaje: str, system: str = '', modelo: str = MODELO_CHAT) -> str:
    mensajes = []
    if system:
        mensajes.append({'role': 'system', 'content': system})
    mensajes.append({'role': 'user', 'content': mensaje})

    resp = client.chat.completions.create(
        model=modelo,
        messages=mensajes,
        max_tokens=512,
        temperature=0.3,
    )
    return resp.choices[0].message.content

respuesta = chat(
    '¿Cuáles son las 3 diferencias clave entre GPT-4o y GPT-4o-mini?',
    system='Responde en español, de forma concisa y estructurada.'
)
print(respuesta)

## 2. Streaming

In [ ]:
import sys

def chat_streaming(mensaje: str, system: str = '') -> str:
    mensajes = []
    if system:
        mensajes.append({'role': 'system', 'content': system})
    mensajes.append({'role': 'user', 'content': mensaje})

    texto_completo = ''
    with client.chat.completions.stream(
        model=MODELO_CHAT,
        messages=mensajes,
        max_tokens=256,
    ) as stream:
        for chunk in stream:
            delta = chunk.choices[0].delta.content or ''
            texto_completo += delta
            print(delta, end='', flush=True)

    print()  # salto de línea final
    return texto_completo

respuesta = chat_streaming('Explica en 3 frases qué es un transformer en ML.')

## 3. Function calling (tool use)

In [ ]:
import json

HERRAMIENTAS = [
    {
        'type': 'function',
        'function': {
            'name': 'obtener_precio_accion',
            'description': 'Obtiene el precio actual de una acción.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'ticker': {'type': 'string', 'description': 'Símbolo bursátil, ej: AAPL, MSFT'},
                    'moneda': {'type': 'string', 'enum': ['USD', 'EUR'], 'default': 'USD'},
                },
                'required': ['ticker'],
            },
        },
    },
    {
        'type': 'function',
        'function': {
            'name': 'calcular_rendimiento',
            'description': 'Calcula el rendimiento porcentual entre dos precios.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'precio_inicial': {'type': 'number'},
                    'precio_final': {'type': 'number'},
                },
                'required': ['precio_inicial', 'precio_final'],
            },
        },
    },
]

# Simulador de herramientas
PRECIOS_MOCK = {'AAPL': 189.5, 'MSFT': 415.2, 'GOOGL': 175.8}

def ejecutar_herramienta(nombre: str, args: dict) -> str:
    if nombre == 'obtener_precio_accion':
        precio = PRECIOS_MOCK.get(args['ticker'], 0)
        moneda = args.get('moneda', 'USD')
        return json.dumps({'ticker': args['ticker'], 'precio': precio, 'moneda': moneda})
    elif nombre == 'calcular_rendimiento':
        rend = ((args['precio_final'] - args['precio_inicial']) / args['precio_inicial']) * 100
        return json.dumps({'rendimiento_pct': round(rend, 2)})
    return json.dumps({'error': 'Herramienta no encontrada'})

def agente_financiero(pregunta: str) -> str:
    mensajes = [{'role': 'user', 'content': pregunta}]

    while True:
        resp = client.chat.completions.create(
            model=MODELO_CHAT,
            messages=mensajes,
            tools=HERRAMIENTAS,
            tool_choice='auto',
        )
        msg = resp.choices[0].message

        if resp.choices[0].finish_reason == 'stop':
            return msg.content

        # Procesar tool calls
        mensajes.append(msg)
        for tc in msg.tool_calls or []:
            resultado = ejecutar_herramienta(tc.function.name, json.loads(tc.function.arguments))
            print(f'  → {tc.function.name}({tc.function.arguments}) = {resultado}')
            mensajes.append({
                'role': 'tool',
                'tool_call_id': tc.id,
                'content': resultado,
            })

print('Pregunta: ¿Cuánto vale AAPL y MSFT hoy, y cuál tiene mejor precio?')
respuesta = agente_financiero('¿Cuánto vale AAPL y MSFT hoy, y cuál tiene mejor precio relativo?')
print('\nRespuesta:', respuesta)

## 4. Embeddings

In [ ]:
import numpy as np

def obtener_embedding(texto: str) -> np.ndarray:
    resp = client.embeddings.create(model=MODELO_EMBED, input=texto)
    return np.array(resp.data[0].embedding)

def obtener_embeddings_lote(textos: list[str]) -> list[np.ndarray]:
    resp = client.embeddings.create(model=MODELO_EMBED, input=textos)
    return [np.array(d.embedding) for d in resp.data]

def similitud_coseno(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Demo de búsqueda semántica
documentos = [
    'Los embeddings representan texto como vectores numéricos.',
    'Python es un lenguaje de programación muy usado en ciencia de datos.',
    'La similitud coseno mide el ángulo entre dos vectores.',
    'ChatGPT es un modelo de lenguaje desarrollado por OpenAI.',
    'NumPy es la librería fundamental para computación numérica en Python.',
]

embs_docs = obtener_embeddings_lote(documentos)

query = '¿Cómo funciona la búsqueda semántica con vectores?'
emb_query = obtener_embedding(query)

scores = [(similitud_coseno(emb_query, emb), doc) for emb, doc in zip(embs_docs, documentos)]
scores.sort(reverse=True)

print(f'Query: "{query}"\n')
print('Resultados por relevancia:')
for score, doc in scores:
    print(f'  {score:.4f} — {doc}')

## 5. Moderación de contenido

In [ ]:
def moderar_texto(texto: str) -> dict:
    resp = client.moderations.create(input=texto)
    resultado = resp.results[0]
    categorias_activas = [
        cat for cat, activa in resultado.categories.__dict__.items() if activa
    ]
    return {
        'flagged': resultado.flagged,
        'categorias_activas': categorias_activas,
        'scores': {k: round(v, 4) for k, v in resultado.category_scores.__dict__.items() if v > 0.01},
    }

textos_prueba = [
    'Me encanta aprender sobre inteligencia artificial.',
    'El modelo de lenguaje generó una respuesta incorrecta.',
]

for texto in textos_prueba:
    resultado = moderar_texto(texto)
    estado = '🔴 BLOQUEADO' if resultado['flagged'] else '✅ OK'
    print(f'{estado} — "{texto[:50]}"')
    if resultado['categorias_activas']:
        print(f'  Categorías: {resultado["categorias_activas"]}')

## 6. Structured output con Pydantic

In [ ]:
from pydantic import BaseModel

class ExtraccionContacto(BaseModel):
    nombre: str
    empresa: str | None
    email: str | None
    telefono: str | None
    cargo: str | None

def extraer_contacto(texto: str) -> ExtraccionContacto:
    resp = client.beta.chat.completions.parse(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': f'Extrae los datos de contacto del siguiente texto:\n\n{texto}'}],
        response_format=ExtraccionContacto,
    )
    return resp.choices[0].message.parsed

texto = """
Hola, me llamo Ana García y trabajo como Directora de Marketing en Acme Corp.
Puedes contactarme en ana.garcia@acme.com o al +34 612 345 678.
"""

contacto = extraer_contacto(texto)
print('Contacto extraído:')
print(f'  Nombre:   {contacto.nombre}')
print(f'  Empresa:  {contacto.empresa}')
print(f'  Email:    {contacto.email}')
print(f'  Teléfono: {contacto.telefono}')
print(f'  Cargo:    {contacto.cargo}')